In [15]:
from pathlib import Path
import pandas as pd

jp_dir = Path(
    r"C:\Users\alys_\Downloads\jpl_files\eviction_cases_2025"
)

jp_files = sorted(
    jp_dir.glob("CasesFiled-*.txt")
)

print("Files found:", len(jp_files))

for f in jp_files:
    print(f.name)

Files found: 12
CasesFiled-01012025To01312025.txt
CasesFiled-02012025To03012025.txt
CasesFiled-03012025To04012025.txt
CasesFiled-04012025To05012025.txt
CasesFiled-05012025To06012025.txt
CasesFiled-06012025To07012025.txt
CasesFiled-07012025To08012025.txt
CasesFiled-08012025To09012025.txt
CasesFiled-09012025To10012025.txt
CasesFiled-10012025To11012025.txt
CasesFiled-11012025To12012025.txt
CasesFiled-12012025To01012026.txt


In [16]:
import csv
import pandas as pd

def read_jp_file_tolerant(path, encoding="utf-8-sig"):

    good_rows = []
    bad_rows = []

    with open(
        path,
        "r",
        encoding=encoding,
        errors="replace",
        newline=""
    ) as f:

        reader = csv.reader(
            f,
            delimiter=",",
            quotechar='"',
            doublequote=True,
            strict=False
        )

        header = next(reader)
        expected_fields = len(header)

        for row in reader:

            if len(row) == expected_fields:
                good_rows.append(row)

            else:
                bad_rows.append({
                    "source_file": path.name,
                    "line_number": reader.line_num,
                    "field_count": len(row),
                    "expected_fields": expected_fields,
                    "row": row
                })

    good_df = pd.DataFrame(
        good_rows,
        columns=header
    ).astype("string")

    good_df["source_file"] = path.name

    bad_df = pd.DataFrame(bad_rows)

    return good_df, bad_df, expected_fields

In [18]:
all_good = []
all_bad = []
file_summary = []

for f in jp_files:

    good, bad, expected = read_jp_file_tolerant(f)

    all_good.append(good)

    if len(bad) > 0:
        all_bad.append(bad)

    file_summary.append({
        "file": f.name,
        "expected_columns": expected,
        "good_rows": len(good),
        "bad_rows": len(bad)
    })

summary = pd.DataFrame(file_summary)

print(summary)

                                 file  expected_columns  good_rows  bad_rows
0   CasesFiled-01012025To01312025.txt                70       6958      6959
1   CasesFiled-02012025To03012025.txt                70       6318      6319
2   CasesFiled-03012025To04012025.txt                70       5734      5735
3   CasesFiled-04012025To05012025.txt                70       5414      5415
4   CasesFiled-05012025To06012025.txt                70       6158      6159
5   CasesFiled-06012025To07012025.txt                70       6863      6866
6   CasesFiled-07012025To08012025.txt                70       6984      6987
7   CasesFiled-08012025To09012025.txt                70       6582      6585
8   CasesFiled-09012025To10012025.txt                70       6841      6848
9   CasesFiled-10012025To11012025.txt                70       6392      6393
10  CasesFiled-11012025To12012025.txt                70       5898      5899
11  CasesFiled-12012025To01012026.txt                70       6497      6500

In [19]:
jp_2025 = pd.concat(
    all_good,
    ignore_index=True
)

print("Parsed rows:", len(jp_2025))
print("Columns:", jp_2025.shape[1])

Parsed rows: 76639
Columns: 71


In [20]:
if all_bad:
    jp_bad_rows = pd.concat(
        all_bad,
        ignore_index=True
    )
else:
    jp_bad_rows = pd.DataFrame()

print("Rows requiring review:", len(jp_bad_rows))

Rows requiring review: 76665


In [21]:
print(summary.to_string(index=False))

print(
    "\nTotal good rows:",
    summary["good_rows"].sum()
)

print(
    "Total malformed rows:",
    summary["bad_rows"].sum()
)

                             file  expected_columns  good_rows  bad_rows
CasesFiled-01012025To01312025.txt                70       6958      6959
CasesFiled-02012025To03012025.txt                70       6318      6319
CasesFiled-03012025To04012025.txt                70       5734      5735
CasesFiled-04012025To05012025.txt                70       5414      5415
CasesFiled-05012025To06012025.txt                70       6158      6159
CasesFiled-06012025To07012025.txt                70       6863      6866
CasesFiled-07012025To08012025.txt                70       6984      6987
CasesFiled-08012025To09012025.txt                70       6582      6585
CasesFiled-09012025To10012025.txt                70       6841      6848
CasesFiled-10012025To11012025.txt                70       6392      6393
CasesFiled-11012025To12012025.txt                70       5898      5899
CasesFiled-12012025To01012026.txt                70       6497      6500

Total good rows: 76639
Total malformed rows: 76665

In [22]:
if len(jp_bad_rows) > 0:
    print(
        jp_bad_rows["field_count"]
        .value_counts()
        .sort_index()
    )

field_count
0     76658
71        4
73        1
74        1
75        1
Name: count, dtype: int64


In [23]:
jp_2025.columns = (
    jp_2025.columns
    .str.strip()
    .str.lower()
    .str.replace(r"\s+", "_", regex=True)
    .str.replace(r"[^a-z0-9_]", "", regex=True)
)

print(jp_2025.columns.tolist())

['case_number', 'jp_court_id', 'case_type', 'case_file_date', 'style_of_case', 'cause_of_action', 'claim_amount', 'case_status', 'plaintiff_name', 'plaintiff_addr_line_1', 'plaintiff_addr_line_2', 'plaintiff_addr_city', 'plaintiff_addr_state', 'plaintiff_addr_zip', 'plaintiff_atty_name', 'plaintiff_atty_addr_1', 'plaintiff_atty_addr_2', 'plaintiff_atty_city', 'plaintiff_atty_state', 'plaintiff_atty_zip', 'defendant_name', 'defendant_addr_line_1', 'defendant_addr_line_2', 'defendant_addr_city', 'defendant_addr_state', 'defendant_addr_zip', 'defendant_atty_name', 'defendant_atty_addr_1', 'defendant_atty_addr_2', 'defendant_atty_city', 'defendant_atty_state', 'defendant_atty_zip', 'second_plaintiff_name', 'second_plaintiff_addr_line_1', 'second_plaintiff_addr_line_2', 'second_plaintiff_addr_city', 'second_plaintiff_addr_state', 'second_plaintiff_addr_zip', 'second_plaintiff_atty_name', 'second_plaintiff_atty_addr_1', 'second_plaintiff_atty_addr_2', 'second_plaintiff_atty_city', 'second_pl

In [24]:
print("Total rows:", len(jp_2025))

print(
    "Unique case numbers:",
    jp_2025["case_number"].nunique()
)

print(
    "Duplicate case numbers:",
    jp_2025["case_number"].duplicated().sum()
)

Total rows: 76639
Unique case numbers: 75516
Duplicate case numbers: 1123


In [25]:
dup_cases = jp_2025[
    jp_2025["case_number"].duplicated(keep=False)
].copy()

print("Rows belonging to duplicated cases:", len(dup_cases))

print(
    "\nNumber of appearances per duplicated case:"
)

print(
    dup_cases["case_number"]
    .value_counts()
    .value_counts()
    .sort_index()
)

Rows belonging to duplicated cases: 2246

Number of appearances per duplicated case:
count
2    1123
Name: count, dtype: int64[pyarrow]


In [26]:
jp_2025 = (
    jp_2025
    .drop_duplicates(
        subset="case_number",
        keep="last"
    )
    .copy()
)

print("Rows after deduplication:", len(jp_2025))
print("Unique cases:", jp_2025["case_number"].nunique())
print("Duplicate case numbers:", jp_2025["case_number"].duplicated().sum())

Rows after deduplication: 75516
Unique cases: 75516
Duplicate case numbers: 0


In [27]:
# Strip whitespace and normalize empty/null strings
for col in jp_2025.columns:
    if jp_2025[col].dtype == "object" or "string" in str(jp_2025[col].dtype):
        jp_2025[col] = (
            jp_2025[col]
            .astype("string")
            .str.strip()
            .replace({
                "": pd.NA,
                "null": pd.NA,
                "NULL": pd.NA,
                "None": pd.NA
            })
        )

# Filing date
jp_2025["case_file_date"] = pd.to_datetime(
    jp_2025["case_file_date"],
    errors="coerce"
)

jp_2025["filing_year"] = jp_2025["case_file_date"].dt.year

In [28]:
print(jp_2025["filing_year"].value_counts(dropna=False).sort_index())
print("Missing filing dates:", jp_2025["case_file_date"].isna().sum())

filing_year
2025    75516
Name: count, dtype: int64
Missing filing dates: 0


In [29]:
print(
    jp_2025["case_type"]
    .value_counts(dropna=False)
    .head(20)
)

print(
    jp_2025["cause_of_action"]
    .value_counts(dropna=False)
    .head(20)
)

case_type
Eviction    75516
Name: count, dtype: int64[pyarrow]
cause_of_action
Eviction                                     74663
<NA>                                           623
Debt Claim                                     102
Small Claims                                    54
Nonpayment - Residential                        45
Foreign Judgment                                15
Driver 's License Suspension Hearing             5
Forcible Entry and Detainer - Residential        2
Holdover - Commercial                            2
Writ of Re-Entry                                 1
Cruelly Treated Animal                           1
Tow Hearing                                      1
Holdover - Residential                           1
Nonpayment - Commercial                          1
Name: count, dtype: int64[pyarrow]


In [30]:
def1 = jp_2025[
    [
        "case_number",
        "case_file_date",
        "filing_year",
        "defendant_name",
        "defendant_addr_line_1",
        "defendant_addr_line_2",
        "defendant_addr_city",
        "defendant_addr_state",
        "defendant_addr_zip"
    ]
].copy()

def1.columns = [
    "case_number",
    "case_file_date",
    "filing_year",
    "defendant_name",
    "addr1",
    "addr2",
    "city",
    "state",
    "zip"
]

def1["defendant_number"] = 1

In [31]:
def2 = jp_2025[
    [
        "case_number",
        "case_file_date",
        "filing_year",
        "second_defendant_name",
        "second_defendant_addr_line_1",
        "second_defendant_addr_line_2",
        "second_defendant_addr_city",
        "second_defendant_addr_state",
        "second_defendant_addr_zip"
    ]
].copy()

def2.columns = [
    "case_number",
    "case_file_date",
    "filing_year",
    "defendant_name",
    "addr1",
    "addr2",
    "city",
    "state",
    "zip"
]

def2["defendant_number"] = 2

In [32]:
jp_addresses = pd.concat(
    [def1, def2],
    ignore_index=True
)

# Keep rows where an address exists
jp_addresses = jp_addresses[
    jp_addresses["addr1"].notna()
].copy()

print("Address rows:", len(jp_addresses))
print("Cases with at least one address:", jp_addresses["case_number"].nunique())

Address rows: 87021
Cases with at least one address: 74116


In [33]:
raw_addresses_per_case = (
    jp_addresses
    .groupby("case_number")["addr1"]
    .nunique(dropna=True)
)

print(
    raw_addresses_per_case
    .value_counts()
    .sort_index()
)

addr1
1    72055
2     2061
Name: count, dtype: int64


In [35]:
import usaddress
import re
jp_addresses["address_raw"] = (
    jp_addresses["addr1"]
    .fillna("")
    .str.strip()
    + " "
    + jp_addresses["addr2"]
    .fillna("")
    .str.strip()
)

jp_addresses["address_raw"] = (
    jp_addresses["address_raw"]
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

In [36]:
def parse_us_address(address):

    result = {
        "house_number": pd.NA,
        "street_predir": pd.NA,
        "street_name": pd.NA,
        "street_suffix": pd.NA,
        "street_postdir": pd.NA,
        "unit_type": pd.NA,
        "unit_number": pd.NA,
        "address_type": pd.NA,
        "parse_success": 0
    }

    if pd.isna(address) or not str(address).strip():
        return result

    try:
        tagged, addr_type = usaddress.tag(str(address))

        result["house_number"] = tagged.get("AddressNumber", pd.NA)
        result["street_predir"] = tagged.get(
            "StreetNamePreDirectional", pd.NA
        )
        result["street_name"] = tagged.get("StreetName", pd.NA)
        result["street_suffix"] = tagged.get(
            "StreetNamePostType", pd.NA
        )
        result["street_postdir"] = tagged.get(
            "StreetNamePostDirectional", pd.NA
        )
        result["unit_type"] = tagged.get(
            "OccupancyType", pd.NA
        )
        result["unit_number"] = tagged.get(
            "OccupancyIdentifier", pd.NA
        )
        result["address_type"] = addr_type
        result["parse_success"] = 1

    except usaddress.RepeatedLabelError:
        pass

    return result

In [39]:
# How many addresses are actually unique?
unique_addresses = (
    jp_addresses["address_raw"]
    .dropna()
    .drop_duplicates()
)

print("Address rows:", len(jp_addresses))
print("Unique addresses to parse:", len(unique_addresses))

parsed_records = []

for address in unique_addresses:
    result = parse_us_address(address)
    result["address_raw"] = address
    parsed_records.append(result)

parsed_unique = pd.DataFrame.from_records(parsed_records)

print(parsed_unique.shape)

Address rows: 87021
Unique addresses to parse: 64337
(64337, 10)


In [40]:
parsed_records = []

for address in unique_addresses:
    result = parse_us_address(address)
    result["address_raw"] = address
    parsed_records.append(result)

parsed_unique = pd.DataFrame.from_records(parsed_records)

print(parsed_unique.shape)

(64337, 10)


In [41]:
jp_addresses = jp_addresses.merge(
    parsed_unique,
    on="address_raw",
    how="left",
    validate="many_to_one"
)

In [43]:
[c for c in jp_addresses.columns if "parse" in c.lower()]

['parse_success_x', 'parse_success_y']

In [44]:
parser_cols = [
    "house_number",
    "street_predir",
    "street_name",
    "street_suffix",
    "street_postdir",
    "unit_type",
    "unit_number",
    "address_type",
    "parse_success"
]

cols_to_drop = []

for col in jp_addresses.columns:
    for base in parser_cols:
        if col == base or col.startswith(base + "_"):
            cols_to_drop.append(col)

print(cols_to_drop)

['house_number_x', 'street_predir_x', 'street_name_x', 'street_suffix_x', 'street_postdir_x', 'unit_type_x', 'unit_number_x', 'address_type_x', 'parse_success_x', 'house_number_y', 'street_predir_y', 'street_name_y', 'street_suffix_y', 'street_postdir_y', 'unit_type_y', 'unit_number_y', 'address_type_y', 'parse_success_y']


In [45]:
jp_addresses = jp_addresses.drop(
    columns=list(set(cols_to_drop)),
    errors="ignore"
)

In [46]:
jp_addresses = jp_addresses.merge(
    parsed_unique,
    on="address_raw",
    how="left",
    validate="many_to_one"
)

In [47]:
print(jp_addresses.shape)

print(
    jp_addresses["parse_success"]
    .value_counts(dropna=False)
)

print(
    "Parse success rate:",
    round(
        jp_addresses["parse_success"].mean() * 100,
        2
    ),
    "%"
)

(87021, 20)
parse_success
1    86372
0      649
Name: count, dtype: int64
Parse success rate: 99.25 %


In [48]:
print(
    jp_addresses[
        [
            "address_raw",
            "house_number",
            "street_predir",
            "street_name",
            "street_suffix",
            "unit_type",
            "unit_number",
            "parse_success"
        ]
    ].head(20)
)

                               address_raw house_number street_predir  \
0            1825 W Little York Rd, Apt 10         1825             W   
1                 4840 N Shepherd Apt 2113         4840             N   
2                       5129 1/2 Weaver Rd         5129           NaN   
3   Apartment at : 1824 Thonig Road Unit 6         1824           NaN   
4        4655 Aldine Mail Route Rd Apt 316         4655           NaN   
5                         4205 Lavender St         4205           NaN   
6                           1718 Viking Dr         1718           NaN   
7   apt #1104 at 4505 Aldine Mail Route Rd         4505           NaN   
8                5605 Chimney Rock Rd #101         5605           NaN   
9                     3919 Essex Lane #135         3919           NaN   
10                           5420 Rand #22         5420           NaN   
11                            5447 Rand #A         5447           NaN   
12                           5431 Rand #A,         

In [49]:
import re
import pandas as pd

def clean_unit_number(x):
    if pd.isna(x):
        return pd.NA

    x = str(x).upper().strip()

    # Remove common symbols/words
    x = re.sub(r"^#\s*", "", x)
    x = re.sub(r"\bAT\b.*$", "", x)
    x = re.sub(r"\s+", " ", x).strip()

    if x == "":
        return pd.NA

    return x


jp_addresses["unit_number_clean"] = (
    jp_addresses["unit_number"]
    .apply(clean_unit_number)
)

In [50]:
print(
    jp_addresses[
        [
            "address_raw",
            "unit_number",
            "unit_number_clean"
        ]
    ].head(20)
)

                               address_raw unit_number unit_number_clean
0            1825 W Little York Rd, Apt 10          10                10
1                 4840 N Shepherd Apt 2113        2113              2113
2                       5129 1/2 Weaver Rd         NaN               NaN
3   Apartment at : 1824 Thonig Road Unit 6           6                 6
4        4655 Aldine Mail Route Rd Apt 316         316               316
5                         4205 Lavender St         NaN               NaN
6                           1718 Viking Dr         NaN               NaN
7   apt #1104 at 4505 Aldine Mail Route Rd   # 1104 at              1104
8                5605 Chimney Rock Rd #101       # 101               101
9                     3919 Essex Lane #135       # 135               135
10                           5420 Rand #22        # 22                22
11                            5447 Rand #A         # A                 A
12                           5431 Rand #A,         

In [51]:
SUFFIX_MAP = {
    "ROAD": "RD",
    "RD.": "RD",
    "STREET": "ST",
    "ST.": "ST",
    "DRIVE": "DR",
    "DR.": "DR",
    "LANE": "LN",
    "LN.": "LN",
    "COURT": "CT",
    "CT.": "CT",
    "AVENUE": "AVE",
    "AVE.": "AVE",
    "BOULEVARD": "BLVD",
    "BLVD.": "BLVD",
    "PARKWAY": "PKWY",
    "HIGHWAY": "HWY",
    "TRAIL": "TRL",
    "CIRCLE": "CIR",
    "PLACE": "PL",
    "TERRACE": "TER"
}

DIR_MAP = {
    "NORTH": "N",
    "SOUTH": "S",
    "EAST": "E",
    "WEST": "W",
    "NORTHEAST": "NE",
    "NORTHWEST": "NW",
    "SOUTHEAST": "SE",
    "SOUTHWEST": "SW"
}


def standardize_component(x, mapping=None):
    if pd.isna(x):
        return ""

    x = str(x).upper().replace(".", "").strip()

    if mapping:
        x = mapping.get(x, x)

    return x

In [52]:
jp_addresses["has_half_address"] = (
    jp_addresses["address_raw"]
    .str.contains(r"\b1\s*/\s*2\b", regex=True, na=False)
    .astype("uint8")
)

In [53]:
jp_addresses["house_number_clean"] = (
    jp_addresses["house_number"]
    .fillna("")
    .astype(str)
    .str.upper()
    .str.strip()
)

jp_addresses.loc[
    jp_addresses["has_half_address"].eq(1),
    "house_number_clean"
] = (
    jp_addresses.loc[
        jp_addresses["has_half_address"].eq(1),
        "house_number_clean"
    ]
    + " 1/2"
)

In [54]:
def make_building_address(row):

    parts = [
        row["house_number_clean"],

        standardize_component(
            row["street_predir"],
            DIR_MAP
        ),

        standardize_component(
            row["street_name"]
        ),

        standardize_component(
            row["street_suffix"],
            SUFFIX_MAP
        ),

        standardize_component(
            row["street_postdir"],
            DIR_MAP
        )
    ]

    parts = [
        p for p in parts
        if p not in ["", "NAN", "NONE"]
    ]

    return " ".join(parts)


jp_addresses["building_address"] = (
    jp_addresses
    .apply(make_building_address, axis=1)
)

In [55]:
print(
    jp_addresses[
        [
            "address_raw",
            "building_address",
            "unit_number_clean"
        ]
    ].head(30)
)

                               address_raw            building_address  \
0            1825 W Little York Rd, Apt 10       1825 W LITTLE YORK RD   
1                 4840 N Shepherd Apt 2113             4840 N SHEPHERD   
2                       5129 1/2 Weaver Rd          5129 1/2 WEAVER RD   
3   Apartment at : 1824 Thonig Road Unit 6              1824 THONIG RD   
4        4655 Aldine Mail Route Rd Apt 316   4655 ALDINE MAIL ROUTE RD   
5                         4205 Lavender St            4205 LAVENDER ST   
6                           1718 Viking Dr              1718 VIKING DR   
7   apt #1104 at 4505 Aldine Mail Route Rd   4505 ALDINE MAIL ROUTE RD   
8                5605 Chimney Rock Rd #101        5605 CHIMNEY ROCK RD   
9                     3919 Essex Lane #135               3919 ESSEX LN   
10                           5420 Rand #22                   5420 RAND   
11                            5447 Rand #A                   5447 RAND   
12                           5431 Rand

In [56]:
normalized_addresses_per_case = (
    jp_addresses[
        jp_addresses["building_address"].ne("")
    ]
    .groupby("case_number")["building_address"]
    .nunique()
)

print(
    normalized_addresses_per_case
    .value_counts()
    .sort_index()
)

building_address
1    73227
2      299
Name: count, dtype: int64


In [57]:
parsed_unique.to_parquet(
    "jp_2025_unique_addresses_parsed.parquet",
    index=False
)

jp_addresses.to_parquet(
    "jp_2025_addresses_clean.parquet",
    index=False
)

jp_2025.to_parquet(
    "jp_2025_cases_clean.parquet",
    index=False
)